# Loan Default Prediction

## 1. Project Overview

### Objective
Build a machine learning workflow to predict whether a borrower will default on a loan.

### Target Variable
- `default`
  - `1` = default
  - `0` = no default

### Notebook Goal
This notebook is a guided template. You will implement the core logic step by step using TODO prompts and hints.

## 2. Imports and Setup

Import core libraries for data handling, visualization, statistics, and modeling.

In [1]:
# TODO: Import common libraries for data science work.
# HINT: pandas, numpy, matplotlib.pyplot, seaborn
# HINT: sklearn modules (model_selection, preprocessing, metrics, linear_model, ensemble)
# HINT: scipy.stats for hypothesis testing

# Example starter imports (expand as needed):
import pandas as pd
import numpy as np

# TODO: Add plotting and ML imports here

# TODO: Set a reproducible random seed for your experiments
random_seed = 42
np.random.seed(random_seed)

## 3. Data Loading

Load your dataset and perform a quick structural inspection before any transformations.

# Loan Default Prediction: Data Dictionary (Revised)

## Purpose
This dictionary standardizes field meaning for EDA, feature engineering, and modeling. Columns are grouped by analytical role.

## 1) Identifiers and Borrower Location
| Column | Description |
|---|---|
| id | A unique LC assigned ID for the loan listing. |
| memberId | A unique LC assigned ID for the borrower member. |
| url | URL for the LC page with listing data. |
| addrState | The state provided by the borrower in the loan application. |
| zip_code | The first 3 numbers of the zip code provided by the borrower in the loan application. |
| msa | Metropolitan Statistical Area of the borrower. |

## 2) Application Timeline and Listing Lifecycle
| Column | Description |
|---|---|
| listD | The date which the borrower's application was listed on the platform. |
| acceptD | The date which the borrower accepted the offer. |
| creditPullD | The date LC pulled credit for this loan. |
| reviewStatus | The status of the loan during the listing period. Values: APPROVED, NOT_APPROVED. |
| reviewStatusD | The date the loan application was reviewed by LC. |
| expD | The date the listing will expire. |
| ils_exp_d | Whole loan platform expiration date. |
| application_type | Indicates whether the loan is an individual application or a joint application with two co-borrowers. |
| initialListStatus | The initial listing status of the loan. Possible values: W, F. |
| disbursement_method | The method by which the borrower receives the loan. Possible values: CASH, DIRECT_PAY. |

## 3) Income, Employment, and Verification
| Column | Description |
|---|---|
| annualInc | The self-reported annual income provided by the borrower during registration. |
| annual_inc_joint | The combined self-reported annual income provided by the co-borrowers during registration. |
| isIncV | Indicates if income was verified by LC, not verified, or if the income source was verified. |
| verified_status_joint | Indicates if the co-borrowers' joint income was verified by LC, not verified, or if the income source was verified. |
| emp_title | The job title supplied by the borrower when applying for the loan. |
| empLength | Employment length in years. Values are between 0 and 10 where 0 means less than one year and 10 means ten or more years. |
| homeOwnership | The home ownership status provided by the borrower during registration. Values: RENT, OWN, MORTGAGE, OTHER. |

## 4) Loan Structure, Pricing, and Assigned Risk
| Column | Description |
|---|---|
| loanAmnt | The listed amount of the loan applied for by the borrower. If credit reduces the amount later, it is reflected here. |
| fundedAmnt | The total amount committed to that loan at that point in time. |
| term | The number of payments on the loan. Values are in months and can be either 36 or 60. |
| intRate | Interest rate on the loan. |
| effective_int_rate | Interest rate on a note reduced by LC's estimate of uncollected interest prior to charge-off. |
| installment | The monthly payment owed by the borrower if the loan originates. |
| grade | LC assigned loan grade. |
| subGrade | LC assigned loan subgrade. |
| serviceFeeRate | Service fee rate paid by the investor for this loan. |
| expDefaultRate | The expected default rate of the loan. |
| purpose | Category provided by the borrower for the loan request. |
| title | Loan title provided by the borrower. |
| desc | Loan description provided by the borrower. |

## 5) Credit Score and Core Debt Burden
| Column | Description |
|---|---|
| ficoRangeLow | The lower boundary range the borrower's FICO at loan origination belongs to. |
| ficoRangeHigh | The upper boundary range the borrower's FICO at loan origination belongs to. |
| dti | Ratio of the borrower's total monthly debt payments (excluding mortgage and requested LC loan) to self-reported monthly income. |
| dti_joint | Ratio of co-borrowers' total monthly debt payments (excluding mortgages and requested LC loan) to combined self-reported monthly income. |

## 6) Utilization, Balances, Limits, and Exposure
| Column | Description |
|---|---|
| all_util | Balance to credit limit on all trades. |
| bcUtil | Ratio of total current balance to high credit/credit limit for all bankcard accounts. |
| il_util | Ratio of total current balance to high credit/credit limit on all installment accounts. |
| revolUtil | Revolving line utilization rate, or the amount of credit used relative to all available revolving credit. |
| percentBcGt75 | Percentage of all bankcard accounts above 75% of limit. |
| bcOpenToBuy | Total open to buy on revolving bankcards. |
| max_bal_bc | Maximum current balance owed on all revolving accounts. |
| revolBal | Total revolving credit balance. |
| total_bal_il | Total current balance of all installment accounts. |
| totalBalExMort | Total credit balance excluding mortgage. |
| totalBcLimit | Total bankcard high credit/credit limit. |
| total_rev_hi_lim | Total revolving high credit/credit limit. |
| total_il_high_credit_limit | Total installment high credit/credit limit. |
| tot_hi_cred_lim | Total high credit/credit limit. |
| avg_cur_bal | Average current balance of all accounts. |
| tot_cur_bal | Total current balance of all accounts. |
| revol_bal_joint | Sum of revolving credit balance of the co-borrowers, net of duplicate balances. |

## 7) Credit Age and Recency
| Column | Description |
|---|---|
| earliestCrLine | The date the borrower's earliest reported credit line was opened. |
| mo_sin_old_rev_tl_op | Months since oldest revolving account opened. |
| mo_sin_rcnt_rev_tl_op | Months since most recent revolving account opened. |
| mo_sin_rcnt_tl | Months since most recent account opened. |
| mths_since_oldest_il_open | Months since oldest bank installment account opened. |
| mths_since_rcnt_il | Months since most recent installment account opened. |
| mthsSinceMostRecentInq | Months since most recent inquiry. |
| mthsSinceRecentBc | Months since most recent bankcard account opened. |
| mthsSinceLastDelinq | Number of months since the borrower's last delinquency. |
| mthsSinceRecentLoanDelinq | Months since most recent personal finance delinquency. |
| mthsSinceRecentRevolDelinq | Months since most recent revolving delinquency. |
| mths_since_last_major_derog | Months since most recent 90-day or worse rating. |
| mthsSinceLastRecord | Number of months since the last public record. |

## 8) Trade and Account Structure Counts
| Column | Description |
|---|---|
| openAcc | Number of open credit lines in the borrower's credit file. |
| totalAcc | Total number of credit lines currently in the borrower's credit file. |
| mortAcc | Number of mortgage accounts. |
| accOpenPast24Mths | Number of trades opened in past 24 months. |
| open_acc_6m | Number of open trades in last 6 months. |
| open_rv_12m | Number of revolving trades opened in past 12 months. |
| open_rv_24m | Number of revolving trades opened in past 24 months. |
| open_il_12m | Number of installment accounts opened in past 12 months. |
| open_il_24m | Number of installment accounts opened in past 24 months. |
| open_act_il | Number of currently active installment trades. |
| num_tl_op_past_12m | Number of accounts opened in past 12 months. |
| num_rev_accts | Number of revolving accounts. |
| num_op_rev_tl | Number of open revolving accounts. |
| num_actv_rev_tl | Number of currently active revolving trades. |
| num_rev_tl_bal_gt_0 | Number of revolving trades with balance > 0. |
| num_bc_tl | Number of bankcard accounts. |
| num_actv_bc_tl | Number of currently active bankcard accounts. |
| num_bc_sats | Number of satisfactory bankcard accounts. |
| num_il_tl | Number of installment accounts. |
| num_sats | Number of satisfactory accounts. |
| total_cu_tl | Number of finance trades. |

## 9) Inquiries, Delinquency, Collections, and Public Derogatory Events
| Column | Description |
|---|---|
| inqLast6Mths | Number of inquiries in past 6 months (excluding auto and mortgage inquiries). |
| inq_last_12m | Number of credit inquiries in past 12 months. |
| inq_fi | Number of personal finance inquiries. |
| delinq2Yrs | Number of 30+ days past-due delinquency incidences in the borrower's credit file for the past 2 years. |
| accNowDelinq | Number of accounts on which the borrower is now delinquent. |
| delinqAmnt | Past-due amount owed for accounts on which the borrower is now delinquent. |
| num_tl_30dpd | Number of accounts currently 30 days past due (updated in past 2 months). |
| num_tl_120dpd_2m | Number of accounts currently 120 days past due (updated in past 2 months). |
| num_tl_90g_dpd_24m | Number of accounts 90 or more days past due in last 24 months. |
| num_accts_ever_120_pd | Number of accounts ever 120 or more days past due. |
| chargeoff_within_12_mths | Number of charge-offs within 12 months. |
| collections_12_mths_ex_med | Number of collections in 12 months excluding medical collections. |
| tot_coll_amt | Total collection amounts ever owed. |
| pct_tl_nvr_dlq | Percent of trades never delinquent. |
| pubRec | Number of derogatory public records. |
| pub_rec_bankruptcies | Number of public record bankruptcies. |
| tax_liens | Number of tax liens. |

## 10) Secondary Applicant (Joint Loans)
| Column | Description |
|---|---|
| sec_app_fico_range_low | FICO range low for the secondary applicant at application. |
| sec_app_fico_range_high | FICO range high for the secondary applicant at application. |
| sec_app_earliest_cr_line | Earliest credit line at time of application for the secondary applicant. |
| sec_app_inq_last_6mths | Credit inquiries in the last 6 months at time of application for the secondary applicant. |
| sec_app_mort_acc | Number of mortgage accounts at time of application for the secondary applicant. |
| sec_app_open_acc | Number of open trades at time of application for the secondary applicant. |
| sec_app_revol_util | Ratio of total current balance to high credit/credit limit for all revolving accounts for the secondary applicant. |
| sec_app_open_act_il | Number of currently active installment trades at time of application for the secondary applicant. |
| sec_app_num_rev_accts | Number of revolving accounts at time of application for the secondary applicant. |
| sec_app_chargeoff_within_12_mths | Number of charge-offs within last 12 months at time of application for the secondary applicant. |
| sec_app_collections_12_mths_ex_med | Number of collections within last 12 months excluding medical collections at time of application for the secondary applicant. |
| sec_app_mths_since_last_major_derog | Months since most recent 90-day or worse rating at time of application for the secondary applicant. |

In [2]:
# Set dataset path
data_path = 'loan.csv'

# Load dataset into a DataFrame
df = pd.read_csv(data_path)

C:\Users\jdblu\AppData\Local\Temp\ipykernel_19944\2672259068.py:5: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [3]:
# Preview dataframe
display(df.head())

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Get dataset shape and target column
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Rows: 2,260,668 | Columns: 145


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), object(36)
memory usage: 2.4+ GB


## 4. Data Cleaning

Identify and handle data quality issues such as missing values, incorrect types, and inconsistent entries.

In [ ]:
# Section 4A: Convert columns to appropriate data types
import re

# Date columns from the revised data dictionary
date_cols = [
    'listD', 'acceptD', 'creditPullD', 'reviewStatusD',
    'expD', 'ils_exp_d', 'earliestCrLine', 'sec_app_earliest_cr_line'
],

                "# Defensive normalization: avoid unhashable errors if a nested list sneaks in",
                "date_cols = [c for c in date_cols if isinstance(c, str)]",
                "",
for col in date_cols:
                "    if isinstance(col, str) and col in df.columns:",
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Common string-to-numeric cleanup for known fields
if 'term' in df.columns:
    df['term'] = (
        df['term']
        .astype(str)
        .str.extract(r'(\d+)', expand=False)
        .pipe(pd.to_numeric, errors='coerce')
        .astype('Int64')
    )

if 'empLength' in df.columns:
    emp_map = {
        '< 1 year': 0,
        '1 year': 1,
        '2 years': 2,
        '3 years': 3,
        '4 years': 4,
        '5 years': 5,
        '6 years': 6,
        '7 years': 7,
        '8 years': 8,
        '9 years': 9,
        '10+ years': 10
    }
    df['empLength'] = (
        df['empLength']
        .replace(emp_map)
        .pipe(pd.to_numeric, errors='coerce')
        .astype('Int64')
    )

percent_cols = ['intRate', 'revolUtil', 'sec_app_revol_util']
for col in percent_cols:
    if col in df.columns and df[col].dtype == 'object':
        cleaned = df[col].astype(str).str.replace('%', '', regex=False).str.strip()
        df[col] = pd.to_numeric(cleaned, errors='coerce')

# Keep these as categorical/text by meaning
categorical_text_cols = [
    'id', 'memberId', 'url', 'addrState', 'zip_code', 'msa', 'application_type',
    'reviewStatus', 'initialListStatus', 'isIncV', 'verified_status_joint',
    'emp_title', 'homeOwnership', 'purpose', 'title', 'desc',
    'grade', 'subGrade', 'disbursement_method'
],

for col in categorical_text_cols:
    if col in df.columns:
        df[col] = df[col].astype('string')

# Try numeric conversion for any remaining object/string columns that are mostly numeric
protected_cols = set(date_cols + categorical_text_cols + ['default'])
for col in df.columns:
    if col in protected_cols:
        continue
    if str(df[col].dtype) in ['object', 'string']:
        trial = pd.to_numeric(df[col].astype(str).str.replace(',', '', regex=False), errors='coerce')
        convert_ratio = trial.notna().mean()
        if convert_ratio >= 0.8:
            df[col] = trial

# Ensure target type is numeric if present
if 'default' in df.columns:
    df['default'] = pd.to_numeric(df['default'], errors='coerce').astype('Int64')

print('Type conversion complete.')
display(df.dtypes.sort_index().head(20))

TypeError: unhashable type: 'list'

In [ ]:
# Section 4B: Build updated feature summary after type conversion
feature_summary = pd.DataFrame({
    'feature': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean().values * 100).round(2),
    'n_unique': df.nunique(dropna=False).values,
})

display(feature_summary.sort_values(['missing_pct', 'n_unique'], ascending=[False, False]).reset_index(drop=True).head(20))

In [ ]:
# Grab columns where the percentage of missing values is greater than a certain threshold (e.g., 50%)
missing_threshold = 50.0
missing_columns = feature_summary.loc[feature_summary["missing_pct"] > missing_threshold, ["feature", "missing_pct"]].sort_values("missing_pct", ascending=False)
print(f"Features with missing_pct > {missing_threshold}%: {len(missing_columns)}")
display(missing_columns.reset_index(drop=True))


Features with missing_pct > 50.0%: 44


,feature,missing_pct
0,id,100.00
1,url,100.00
2,member_id,100.00
3,orig_projected_additional_accrued_interest,99.63
4,hardship_length,99.53
5,hardship_reason,99.53
6,hardship_status,99.53
7,deferral_term,99.53
8,hardship_amount,99.53
9,hardship_start_date,99.53


Columns with greater than 50% of their values missing (null) are generally considered poor candidates for analysis because they contain more missing information than observed data. With such a large proportion of absent values, any patterns or relationships derived from the feature are likely to be unreliable and heavily influenced by imputation assumptions rather than actual data. Additionally, imputing a majority of the values could introduce significant bias and noise, potentially distorting model performance.

In [ ]:
# Section 4C: Identify high-cardinality text columns
non_numeric_cols = df.select_dtypes(exclude=[np.number, 'datetime64[ns]']).columns
non_numeric_summary = feature_summary.loc[
    feature_summary['feature'].isin(non_numeric_cols),
    ['feature', 'dtype', 'n_unique', 'missing_pct']
].sort_values('n_unique', ascending=False)

high_cardinality_threshold = 50
high_cardinality_cols = non_numeric_summary.loc[
    non_numeric_summary['n_unique'] > high_cardinality_threshold, 'feature'
].tolist()

print(f'High-cardinality non-numeric columns (>{high_cardinality_threshold} unique): {len(high_cardinality_cols)}')
display(non_numeric_summary.head(20))
print('Candidate high-cardinality columns:')
print(high_cardinality_cols)


Top 10 non-numeric columns by unique count:


,feature,dtype,n_unique
10,emp_title,object,512695
19,desc,object,124501
21,title,object,63155
22,zip_code,object,957
26,earliest_cr_line,object,755
112,sec_app_earliest_cr_line,object,664
48,last_credit_pull_d,object,141
15,issue_d,object,139
45,last_pymnt_d,object,136
47,next_pymnt_d,object,106


In [ ]:
# Section 4D: Drop columns using missingness, cardinality, and modeling judgment
missing_threshold = 50.0
high_missing_cols = feature_summary.loc[
    feature_summary['missing_pct'] > missing_threshold, 'feature'
].tolist()

# Drop identifiers/unstructured text and leakage-prone column by judgment
manual_drop_cols = [
    'id', 'memberId', 'url', 'desc', 'title', 'emp_title', 'expDefaultRate'
],

columns_to_drop = sorted(set(high_missing_cols + high_cardinality_cols + manual_drop_cols))
columns_to_drop = [c for c in columns_to_drop if c in df.columns]

print(f'Columns selected for drop: {len(columns_to_drop)}')
display(pd.DataFrame({'dropped_column': columns_to_drop}).reset_index(drop=True))

df = df.drop(columns=columns_to_drop)

print(f'Updated dataset shape after dropping columns: {df.shape}')
print('Remaining dtypes (top 20):')
display(df.dtypes.sort_index().head(20))


Categorical fields with n_unique <= 15:


,feature,dtype,n_unique
0,term,object,2
1,disbursement_method,object,2
2,hardship_type,object,2
3,hardship_flag,object,2
4,debt_settlement_flag,object,2
5,initial_list_status,object,2
6,application_type,object,2
7,pymnt_plan,object,2
8,verification_status,object,3
9,verification_status_joint,object,4


import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})

# Identify column families after dropping in Section 4
numeric_cols   = df.select_dtypes(include=[np.number]).columns.tolist()
datetime_cols  = df.select_dtypes(include=['datetime64[ns]']).columns.tolist()
cat_cols       = df.select_dtypes(include=['object', 'string']).columns.tolist()

# Exclude target from feature lists
TARGET = 'default'
num_features = [c for c in numeric_cols if c != TARGET]
cat_features = [c for c in cat_cols if c != TARGET]

print(f'Numeric features : {len(num_features)}')
print(f'Categorical features : {len(cat_features)}')
print(f'Datetime features : {len(datetime_cols)}')


### 5.1 Target Variable Distribution

Examine class balance before any modelling. A strong imbalance will influence the choice of evaluation metric (favouring precision/recall/AUC over raw accuracy) and may require class-weighting or resampling.


In [ ]:
# 5.1 Target Variable Distribution
target_counts = df[TARGET].value_counts().sort_index()
target_pcts   = df[TARGET].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
labels = ['No Default (0)', 'Default (1)']
colors = ['#5DA5DA', '#F15854']
bars = axes[0].bar(labels, target_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Target Class — Count')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)

# Pie
axes[1].pie(
    target_pcts.values,
    labels=[f'No Default\n{target_pcts.iloc[0]:.1f}%', f'Default\n{target_pcts.iloc[1]:.1f}%'],
    colors=colors, autopct='%1.1f%%', startangle=90
)
axes[1].set_title('Target Class — Proportion')

plt.suptitle(f'Target Variable Distribution: `{TARGET}`', fontsize=14)
plt.tight_layout()
plt.show()

imbalance_ratio = target_counts.max() / target_counts.min()
print(f'Class counts:\n{target_counts.to_string()}')
print(f'\nClass proportions (%):\n{target_pcts.round(2).to_string()}')
print(f'\nImbalance ratio (majority:minority): {imbalance_ratio:.2f}:1')
print('\n> Note: if the ratio is > 3:1, consider stratified splitting and class-weighted models.')


### 5.2 Univariate Analysis: Numeric Features

Examine descriptive statistics and distributions for all numeric features. High skewness (|skew| > 2) flags candidates for log-transformation during feature engineering. Kurtosis > 3 suggests heavy tails and potential outlier sensitivity.


In [ ]:
# 5.2a Descriptive statistics for numeric features
num_stats = (
    df[num_features]
    .describe(percentiles=[.05, .25, .5, .75, .95])
    .T
)
num_stats['skew']     = df[num_features].skew()
num_stats['kurt']     = df[num_features].kurtosis()
num_stats['missing%'] = (df[num_features].isna().mean() * 100).round(2)

display_cols = ['count', 'mean', 'std', 'min', '5%', '25%', '50%', '75%', '95%', 'max', 'skew', 'kurt', 'missing%']
print(f'Numeric features: {len(num_features)}  — sorted by |skew|')
display(num_stats[display_cols].sort_values('skew', key=abs, ascending=False).head(30))


In [ ]:
# 5.2b Histogram grid for top numeric features
plot_candidates = [
    c for c in num_features
    if df[c].nunique() > 5 and df[c].notna().mean() >= 0.3
]
plot_numeric = plot_candidates[:20]

# Used in later sections (5.5) — store top 8 by data coverage
top_numeric = sorted(plot_candidates[:8], key=lambda c: df[c].notna().sum(), reverse=True)

n_cols = 4
n_rows = -(-len(plot_numeric) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(plot_numeric):
    data = df[col].dropna()
    lo, hi = data.quantile(0.01), data.quantile(0.99)
    axes[i].hist(data.clip(lo, hi), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Univariate Distributions — Numeric Features (1st–99th pct clip)', fontsize=14)
plt.tight_layout()
plt.show()

# Flag highly skewed features
skew_series = df[plot_numeric].skew().abs().sort_values(ascending=False)
high_skew = skew_series[skew_series > 2]
print(f'\nHighly skewed features (|skew| > 2): {len(high_skew)}')
display(high_skew.reset_index().rename(columns={'index': 'feature', 0: 'abs_skew'}))


### 5.3 Univariate Analysis: Categorical Features

Examine value distributions for categorical features. Low-cardinality fields are plotted as bar charts; high-cardinality fields are summarised in a table showing top value frequency.


In [ ]:
# 5.3 Univariate Categorical — value counts and bar charts
low_card_cats = [c for c in cat_features if df[c].nunique() <= 15]

if low_card_cats:
    n_cols = 2
    n_rows = -(-len(low_card_cats) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).flatten()

    for i, col in enumerate(low_card_cats):
        counts = df[col].value_counts(dropna=False).sort_values(ascending=False)
        axes[i].bar(counts.index.astype(str), counts.values,
                    color=sns.color_palette('muted', len(counts)))
        axes[i].set_title(col)
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].set_ylabel('Count')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Univariate Distribution — Categorical Features', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No low-cardinality categorical features found after column dropping.')

# Summary table
print('\nCategorical feature summary:')
display(pd.DataFrame({
    'feature':      cat_features,
    'n_unique':     [df[c].nunique(dropna=False) for c in cat_features],
    'top_value':    [df[c].mode().iloc[0] if not df[c].mode().empty else None for c in cat_features],
    'top_freq_pct': [(df[c].value_counts(normalize=True).iloc[0] * 100).round(2)
                     if df[c].notna().any() else None for c in cat_features],
}).sort_values('n_unique'))


### 5.4 Bivariate Analysis

Examine relationships between pairs of key numeric features, coloured by default class. Joint separability between groups suggests useful interaction terms or non-linear boundaries.


In [ ]:
# 5.4 Bivariate Analysis — scatter plots coloured by default class
bivariate_pairs = [
    ('loanAmnt', 'annualInc'),
    ('intRate',  'dti'),
    ('revolUtil', 'ficoRangeLow'),
]
valid_pairs = [(a, b) for a, b in bivariate_pairs if a in df.columns and b in df.columns]

if valid_pairs:
    fig, axes = plt.subplots(1, len(valid_pairs), figsize=(6 * len(valid_pairs), 5))
    if len(valid_pairs) == 1:
        axes = [axes]
    for ax, (x_col, y_col) in zip(axes, valid_pairs):
        sample = (
            df[[x_col, y_col, TARGET]].dropna()
            .sample(min(3000, len(df)), random_state=42)
        )
        sc = ax.scatter(
            sample[x_col], sample[y_col],
            c=sample[TARGET].astype(float),
            cmap='coolwarm', alpha=0.35, s=10
        )
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title(f'{x_col} vs {y_col}')
        plt.colorbar(sc, ax=ax, label='default')
    plt.suptitle('Bivariate Scatter Plots (coloured by Default)', fontsize=13)
    plt.tight_layout()
    plt.show()

# Pairplot for a small subset of key features
pairplot_cols = [c for c in ['loanAmnt', 'intRate', 'dti', 'revolUtil', 'ficoRangeLow', TARGET]
                 if c in df.columns]
sample_pp = df[pairplot_cols].dropna().sample(min(1500, len(df)), random_state=42).copy()
sample_pp[TARGET] = sample_pp[TARGET].astype(int).astype(str)

g = sns.pairplot(
    sample_pp, hue=TARGET,
    diag_kind='kde',
    plot_kws={'alpha': 0.3, 's': 12},
    palette={'0': '#5DA5DA', '1': '#F15854'}
)
g.fig.suptitle('Pairplot of Key Features (coloured by Default)', y=1.02, fontsize=13)
plt.show()


### 5.5 Features Compared to Target Variable

Compare numeric feature distributions across the two default classes using boxplots and a mean-difference table. Features with large group-level differences are strong candidate predictors.


In [ ]:
# 5.5a Boxplots — top numeric features by default class
n_plot = min(8, len(top_numeric))
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(top_numeric[:n_plot]):
    sns.boxplot(
        data=df[df[col].notna()],
        x=TARGET, y=col,
        palette={0: '#5DA5DA', 1: '#F15854'},
        showfliers=False,
        ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('Default (0=No, 1=Yes)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Key Numeric Features vs Default Class (outliers hidden)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# 5.5b Mean feature values by default class
mean_by_target = df.groupby(TARGET)[top_numeric].mean().T
mean_by_target.columns = ['Non-Default (0)', 'Default (1)']
mean_by_target['Abs % Diff'] = (
    (mean_by_target['Default (1)'] - mean_by_target['Non-Default (0)'])
    / mean_by_target['Non-Default (0)'].replace(0, np.nan)
    * 100
).abs().round(2)

print('Mean feature values by default class — sorted by absolute % difference:')
display(mean_by_target.sort_values('Abs % Diff', ascending=False))


### 5.6 Correlation Matrix

Compute pairwise Pearson correlations across numeric features. Focus on the top features correlated with `default` to guide feature selection, and flag pairs with very high mutual correlation as potential multicollinearity candidates for linear models.


In [ ]:
# 5.6 Correlation Matrix
# Use numeric features with sufficient data coverage
corr_candidates = [c for c in num_features if df[c].notna().mean() >= 0.5]
corr_df = df[corr_candidates + [TARGET]].corr()

# Rank features by absolute correlation with target, take top 25
target_corr = corr_df[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top25 = target_corr.head(25).index.tolist()
plot_corr = df[top25 + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(17, 14))
mask = np.triu(np.ones_like(plot_corr, dtype=bool))
sns.heatmap(
    plot_corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, ax=ax,
    annot_kws={'size': 7}
)
ax.set_title(f'Correlation Matrix — Top 25 Features Most Correlated with `{TARGET}`', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nTop 15 features by |correlation| with `{TARGET}`:')
display(
    target_corr.head(15)
    .reset_index()
    .rename(columns={'index': 'feature', TARGET: 'abs_corr_with_default'})
)


### 5.7 Additional EDA: Default Rates by Category and Loan Characteristics

Examine how default rates vary across loan grade, purpose, and other categorical splits. Also inspect the relationship between interest rate and FICO score as a data quality and risk-pricing check.


In [ ]:
# 5.7 Additional EDA: Default rate by grade and purpose
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if 'grade' in df.columns:
    grade_default = df.groupby('grade')[TARGET].mean().sort_index()
    axes[0].bar(grade_default.index, grade_default.values,
                color=sns.color_palette('muted', len(grade_default)))
    axes[0].set_title('Default Rate by Loan Grade')
    axes[0].set_xlabel('Grade')
    axes[0].set_ylabel('Default Rate')
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

if 'purpose' in df.columns:
    purpose_default = df.groupby('purpose')[TARGET].mean().sort_values(ascending=False)
    purpose_default.plot(kind='bar', ax=axes[1],
                         color=sns.color_palette('muted', len(purpose_default)))
    axes[1].set_title('Default Rate by Loan Purpose')
    axes[1].set_xlabel('Purpose')
    axes[1].set_ylabel('Default Rate')
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Loan amount distribution by grade
if 'loanAmnt' in df.columns and 'grade' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    grade_order = sorted(df['grade'].dropna().unique())
    sns.boxplot(data=df, x='grade', y='loanAmnt', order=grade_order, palette='muted', ax=ax)
    ax.set_title('Loan Amount Distribution by Grade')
    ax.set_xlabel('Grade')
    ax.set_ylabel('Loan Amount ($)')
    plt.tight_layout()
    plt.show()

# Interest rate vs FICO by default class
if 'intRate' in df.columns and 'ficoRangeLow' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, color in [(0, '#5DA5DA'), (1, '#F15854')]:
        sub = df[df[TARGET] == label][['intRate', 'ficoRangeLow']].dropna().sample(min(2000, len(df)), random_state=42)
        ax.scatter(sub['ficoRangeLow'], sub['intRate'], alpha=0.3, s=8, color=color,
                   label='No Default' if label == 0 else 'Default')
    ax.set_xlabel('FICO Range Low')
    ax.set_ylabel('Interest Rate (%)')
    ax.set_title('Interest Rate vs FICO Score by Default Class')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# TODO: Analyze target distribution and class imbalance
# HINT: df['default'].value_counts(normalize=True)

# TODO: Univariate analysis for numeric features
# HINT: hist(), boxplot(), describe()

# TODO: Univariate analysis for categorical features
# HINT: value_counts(), seaborn.countplot

# TODO: Compare features by target class
# HINT: groupby('default'), seaborn.boxplot/histplot

## 6. Statistical Testing

Use statistical hypothesis tests to assess whether observed group differences are likely meaningful.

Interpretation reminder:
- Define a significance level (for example, alpha = 0.05).
- Compare p-value against alpha.
- Explain practical meaning, not only statistical significance.

### 6.1 T-Tests

Compare mean values of a numeric feature between default and non-default groups.

In [ ]:
# TODO: Select a numeric feature to test
feature_name = 'your_numeric_feature'

# TODO: Split feature values by target class
# HINT: df[df['default'] == 1][feature_name] and df[df['default'] == 0][feature_name]

# TODO: Run independent t-test
# HINT: from scipy.stats import ttest_ind
# HINT: t_stat, p_value = ttest_ind(group_default, group_non_default, equal_var=False)

# TODO: Interpret p-value in markdown
# TODO: State whether you reject or fail to reject the null hypothesis

### 6.2 ANOVA

Compare means across more than two groups for a numeric outcome.

In [ ]:
# TODO: Choose a grouping feature with 3+ categories and a numeric measurement
group_col = 'your_group_column'
value_col = 'your_numeric_feature'

# TODO: Build group arrays for ANOVA
# HINT: [df[df[group_col] == g][value_col].dropna() for g in df[group_col].unique()]

# TODO: Run ANOVA
# HINT: from scipy.stats import f_oneway
# HINT: f_stat, p_value = f_oneway(*group_arrays)

# TODO: Interpret p-value and discuss which post-hoc analysis you might run next

### 6.3 Chi-Square Test

Assess independence between two categorical variables.

In [ ]:
# TODO: Choose two categorical columns
cat_col_1 = 'your_categorical_feature_1'
cat_col_2 = 'default'

# TODO: Build a contingency table
# HINT: pd.crosstab(df[cat_col_1], df[cat_col_2])

# TODO: Run chi-square test
# HINT: from scipy.stats import chi2_contingency
# HINT: chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# TODO: Interpret p-value and explain whether variables appear independent

## 7. Feature Engineering

Create model-ready features through transformation, encoding, and scaling.

In [ ]:
# TODO: Create new features (example: financial ratios)
# HINT: debt_to_income = debt / income (handle division by zero carefully)

# TODO: Encode categorical variables
# HINT: pd.get_dummies() or sklearn.preprocessing.OneHotEncoder

# TODO: Scale numeric variables where appropriate
# HINT: sklearn.preprocessing.StandardScaler

# TODO: Keep a record of feature transformations for reproducibility

## 8. Train-Test Split

Split your engineered dataset into train and test sets before model training.

In [ ]:
# TODO: Separate features and target
# HINT: X = ..., y = df['default']

# TODO: Perform train-test split
# HINT: from sklearn.model_selection import train_test_split
# HINT: Use stratify=y if class imbalance exists

X_train, X_test, y_train, y_test = None, None, None, None

## 9. Baseline Model (Logistic Regression)

Train a baseline classifier and generate initial predictions.

In [ ]:
# TODO: Initialize logistic regression model
# HINT: from sklearn.linear_model import LogisticRegression

# TODO: Fit model on training data

# TODO: Generate class predictions and prediction probabilities
# HINT: model.predict(X_test), model.predict_proba(X_test)[:, 1]

baseline_model = None
y_pred_log = None
y_proba_log = None

## 10. Regularization

Compare L1 and L2 regularized logistic regression to evaluate sparsity and stability effects.

In [ ]:
# TODO: Train L1-regularized logistic regression
# HINT: penalty='l1', choose compatible solver (for example, 'liblinear' or 'saga')

# TODO: Train L2-regularized logistic regression
# HINT: penalty='l2'

# TODO: Compare coefficients between L1 and L2 models
# HINT: Inspect model.coef_ values and count near-zero coefficients

l1_model = None
l2_model = None

## 11. Tree-Based Models

Train nonlinear models and compare them with logistic regression baseline.

In [ ]:
# TODO: Train Random Forest classifier
# HINT: from sklearn.ensemble import RandomForestClassifier

# TODO (Optional): Train Gradient Boosting classifier
# HINT: from sklearn.ensemble import GradientBoostingClassifier

rf_model = None
gb_model = None  # Optional

## 12. Model Evaluation

Evaluate model quality with confusion matrix and classification metrics focused on default-risk use cases.

In [ ]:
# TODO: Compute confusion matrix for each model
# HINT: sklearn.metrics.confusion_matrix

# TODO: Calculate precision, recall, and F1
# HINT: sklearn.metrics.classification_report or precision_score/recall_score/f1_score

# TODO: Compare where false negatives vs false positives matter most for this problem

## 13. ROC Curve and AUC

### What ROC Represents
The ROC curve shows the trade-off between True Positive Rate and False Positive Rate across classification thresholds.

### Why AUC Is Useful
AUC summarizes ranking performance across all thresholds into one score. Higher AUC generally indicates better class separation.

In [ ]:
# TODO: Compute ROC curve coordinates
# HINT: from sklearn.metrics import roc_curve

# TODO: Compute AUC score
# HINT: from sklearn.metrics import roc_auc_score

# TODO: Plot ROC curves for multiple models on the same figure
# HINT: matplotlib.pyplot.plot

## 14. Feature and Hyperparameter Selection

Define a strategy to select the most useful features and tune key model hyperparameters in a reproducible way.

Suggested approach:
- Start with a small set of candidate models.
- Use cross-validation to compare settings.
- Track both model performance and interpretability.

### Tuning Run Log (Template)
Use this compact table to record each run.

| Run ID | Model | Feature Set | Search Method | Hyperparameters Tested | CV Metric | Mean CV Score | Validation/Test Score | Notes |
|---|---|---|---|---|---|---|---|---|
| 1 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

In [ ]:
# TODO: Define candidate feature subsets
# HINT: Try baseline feature set vs engineered feature set

# TODO: Define hyperparameter grids for one or more models
# HINT: Use sklearn.model_selection.GridSearchCV or RandomizedSearchCV
# HINT: Tune examples: C (logistic regression), max_depth/n_estimators (random forest)

# TODO: Choose a scoring metric aligned with business goals
# HINT: precision, recall, f1, or roc_auc

# TODO: Run cross-validated search on training data only
# TODO: Record best params and best cross-validation score for each model

best_params_summary = None


## 15. Model Comparison

Create a side-by-side summary of all candidate models and select a final approach.

In [ ]:
# TODO: Build a comparison table for key metrics
# HINT: Include Precision, Recall, F1, AUC, and notes on interpretability

# TODO: Select the best model based on project priorities
# TODO: Justify your selection in 2-4 concise bullet points

## 16. Feature Importance and Interpretation

Interpret which features most strongly influence predicted default risk.

In [ ]:
# TODO: Extract feature importance or coefficient-based influence
# HINT: RandomForestClassifier.feature_importances_
# HINT: LogisticRegression.coef_ (check sign and magnitude)

# TODO: Visualize top features
# HINT: bar chart with sorted importance values

# TODO: Interpret key drivers in plain business language

## 17. Final Insights and Business Recommendations

### TODO: Summarize Findings
- What patterns were most predictive of default?
- Which model performed best and why?

### TODO: Translate to Business Impact
- How could this model support lending decisions?
- What is the trade-off between catching defaults and rejecting safe applicants?
- What monitoring or retraining strategy would you propose for production use?